# Deep Dive Notes: The Variational Autoencoder (VAE)

---

## 0. Prerequisites: The Mathematical Toolkit

To understand *why* VAEs are built the way they are, we must first establish specific tools from Calculus and Statistics.

### 0.1. Calculus Concepts
* **The Gradient ($\nabla$):** A vector pointing in the direction of the steepest increase of a function. In Deep Learning, we compute the gradient of the Loss Function with respect to the weights ($\nabla_\theta \mathcal{L}$) to know how to update the network.
* **The Chain Rule:** The rule for differentiating composite functions.
    * If $z = f(y)$ and $y = g(x)$, then $\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$.
    * **VAE Application:** This is crucial for **Backpropagation**. It allows the error signal to travel from the output layer, through the decoder, through the latent sampling step (via reparameterization), back to the encoder.
* **Intractable Integrals:** An integral is "intractable" if it cannot be computed in a reasonable amount of time (e.g., requires summing over infinite possibilities).
    * **VAE Application:** Computing the true probability of data, $P(x) = \int P(x|z)P(z)dz$, is intractable. This forces us to use **Variational Inference** (approximation) instead of exact calculation.

### 0.2. Statistics & Probability Concepts
* **Probability Distribution:** A mathematical function that gives the probabilities of occurrence of different possible outcomes for an experiment.
    * **VAE Application:** We treat the latent code $z$ not as a number, but as a distribution $P(z)$.
* **Gaussian (Normal) Distribution $\mathcal{N}(\mu, \sigma^2)$:** The "Bell Curve," defined by its center ($\mu$) and spread ($\sigma$).
* **Conditional Probability $P(A|B)$:** The probability of A happening given that B has occurred.
    * **Encoder:** $Q_\phi(z|x)$ — Probability of latent code $z$ given input image $x$.
    * **Decoder:** $P_\theta(x|z)$ — Probability of image $x$ given latent code $z$.
* **Expectation $\mathbb{E}$:** The long-run average value of a random variable.
    * **VAE Application:** The reconstruction loss is an expectation: "On average, how close is the output to the input?"
* **KL Divergence ($D_{KL}$):** A metric to measure how different two probability distributions are.
    * **VAE Application:** Used as a **Regularizer**. It forces the encoder's learned distribution $Q(z|x)$ to look like a standard Normal distribution $\mathcal{N}(0,1)$.
* **Log-Likelihood:** The natural logarithm of the probability. We use logs because multiplying many small probabilities causes numerical underflow (values turn to 0). Logs turn multiplication into addition ($\log(ab) = \log a + \log b$), which is numerically stable.

### 0.3. The Bridge: Jensen's Inequality
This theorem connects calculus and statistics to create the VAE objective function.
For any concave function $f$ (like $\log$), the **function of the average** is greater than or equal to the **average of the function**.

$$
f(\mathbb{E}[x]) \ge \mathbb{E}[f(x)]
$$

Specifically for logarithms:

$$
\log(\mathbb{E}[y]) \ge \mathbb{E}[\log(y)]
$$

---

## 1. The Standard Autoencoder (Deterministic)

### 1.1. The Architecture
The Standard Autoencoder is a dimensionality reduction engine composed of two networks:
1.  **Encoder ($f_\phi$):** Compresses high-dimensional input $x$ into a low-dimensional bottleneck $z$.
    $$z = f_\phi(x)$$
2.  **Decoder ($g_\theta$):** Attempts to reconstruct $x$ from $z$.
    $$\hat{x} = g_\theta(z)$$

### 1.2. The Objective
Minimize the **Reconstruction Loss** (typically Mean Squared Error):

$$
\mathcal{L} = || x - \hat{x} ||^2
$$

### 1.3. The Limitation
This system is **Deterministic**.
* Input A always produces exactly Code A.
* Input B always produces exactly Code B.
* The model does not learn the *structure* of the data, only how to compress specific points.

---

## 2. The Latent Space Problem

### 2.1. Discontinuity ("The Dead Zone")
Because the standard AE minimizes reconstruction error for specific training points, it creates a "discontinuous" latent space.
* **Cluster Formation:** It might map all "Dogs" to coordinate `[-5, 5]` and all "Cats" to `[5, 5]`.
* **The Void:** The space in between (e.g., `[0, 0]`) is undefined. The decoder has never seen a code there.
* **Consequence:** Sampling from the gap produces garbage noise. The model cannot interpolate or generate new, valid data.

---

## 3. The VAE Solution (Probabilistic)

### 3.1. The Paradigm Shift
To fix the discontinuity, we fundamentally change the encoder. Instead of outputting a single point $z$, the Encoder outputs parameters for a **Probability Distribution**:
1.  **Mean ($\mu$):** The center of the latent cloud.
2.  **Variance ($\sigma^2$):** The size/spread of the latent cloud.

### 3.2. Sampling
We generate the latent vector $z$ by sampling from this distribution:

$$
z \sim \mathcal{N}(\mu, \sigma^2)
$$

### 3.3. Why this works
By introducing noise ($\sigma$) during the training process, we force the decoder to be robust. It learns that **every point** within the standard deviation of $\mu$ should reconstruct to the original input. This "smears" the data points into overlapping regions, eliminating the dead zones and allowing smooth interpolation.

---

## 4. The Mathematics (Deep Dive)

We want to generate data. To do this, we maximize the **Marginal Log-Likelihood** of the data, $\log P(x)$.

### 4.1. The Derivation of the ELBO

**Step 1: The Intractable Integral**
$$P(x) = \int P(x|z)P(z) \, dz$$
We cannot compute this integral because checking every possible $z$ is impossible.

**Step 2: Variational Inference (The "Multiply by 1" Trick)**
We introduce an approximate posterior $Q(z|x)$ (our Encoder) and multiply/divide by it:
$$\log P(x) = \log \int P(x, z) \frac{Q(z|x)}{Q(z|x)} \, dz$$

**Step 3: Definition of Expectation**
$$\log P(x) = \log \mathbb{E}_{z \sim Q} \left[ \frac{P(x, z)}{Q(z|x)} \right]$$

**Step 4: Applying Jensen's Inequality**
We move the $\log$ inside the expectation (switching $=$ to $\ge$). This gives us a lower bound to optimize.
$$\log P(x) \ge \mathbb{E}_{z \sim Q} \left[ \log \frac{P(x, z)}{Q(z|x)} \right]$$

**Step 5: Rearranging to ELBO**
Using logarithm rules ($\log \frac{a}{b} = \log a - \log b$) and probability definitions ($P(x,z) = P(x|z)P(z)$):

$$
\text{ELBO} = \mathbb{E}_{z \sim Q}[\log P(x|z)] - D_{KL}(Q(z|x) || P(z))
$$

### 4.2. The VAE Loss Function
The final loss function derived from the ELBO has two competing terms:

1.  **Reconstruction Loss ($\mathbb{E}[\log P(x|z)]$):**
    * **Goal:** Accuracy.
    * **Action:** Minimizes the difference between input and output.
    * **Effect:** Tries to reduce variance $\sigma$ to 0 (collapsing back to a standard AE).

2.  **KL Divergence ($D_{KL}$):**
    * **Goal:** Regularization.
    * **Action:** Measures the distance between the learned distribution $Q$ and a standard Unit Gaussian $\mathcal{N}(0, I)$.
    * **Effect:** Forces the latent clouds to be spherical and centered at 0, preventing them from drifting too far apart.

---

## 5. The Reparameterization Trick (Calculus)

### 5.1. The Gradient Blockage
Neural networks learn via backpropagation (Chain Rule).
* To train the encoder, we need gradients to flow through the sampling node $z$.
* However, $z$ is the result of a random process (stochastic sampling).
* **Problem:** You cannot differentiate a random variable. $\frac{\partial (\text{Random})}{\partial \phi}$ is undefined. The gradient chain breaks.

### 5.2. The Solution
We use the **Reparameterization Trick** to move the randomness out of the network's direct path.

We rewrite the random variable $z$ as a deterministic function of inputs and fixed noise:

$$
z = \mu + \sigma \odot \epsilon
$$

$$
\text{where } \epsilon \sim \mathcal{N}(0, 1)
$$

### 5.3. Calculus Proof of Flow
Now, $z$ is just a function $f(\mu, \sigma, \epsilon)$. We can compute partial derivatives with respect to the encoder's outputs:

* **Derivative w.r.t Mean:** $\frac{\partial z}{\partial \mu} = 1$
* **Derivative w.r.t Std Dev:** $\frac{\partial z}{\partial \sigma} = \epsilon$

Since $\epsilon$ is external fixed noise, it is treated as a constant during differentiation. This restores the connection for the Chain Rule, allowing gradients to flow from the Loss $\to$ Decoder $\to$ $z$ $\to$ Encoder ($\mu, \sigma$).

In [20]:
from manim import *
import numpy as np

# --- Visual Style Constants ---
COLOR_INPUT = BLUE
COLOR_LATENT = YELLOW
COLOR_OUTPUT = GREEN
COLOR_ENCODER = TEAL
COLOR_DECODER = MAROON
COLOR_MATH = WHITE
COLOR_NOISE = RED_C
TEXT_COLOR = LIGHT_GRAY

class AutoencoderDeepDive(Scene):
    def construct(self):
        # 1. Standard AE: Detailed Compression Flow
        self.next_section("Introduction to AE")
        self.part_1_standard_autoencoder()
        
        # 2. Latent Space: Probing the Void
        self.next_section("The Latent Space Problem")
        self.wait(1)
        self.part_2_latent_space_problem()
        
        # 3. VAE Solution: The Sampling Process
        self.next_section("Intro to VAE")
        self.wait(1)
        self.part_3_vae_concept()
        
        # 4. Math: Visualizing the Balance
        self.next_section("VAE Mathematics Deep Dive")
        self.wait(1)
        self.part_4_vae_mathematics_deep()
        
        # 5. Calculus: The Gradient Path
        self.next_section("Reparameterization Trick")
        self.wait(1)
        self.part_5_reparameterization_deep()
        
        self.wait(3)

    def part_1_standard_autoencoder(self):
        # Title Sequence
        title = Title("Part 1: The Standard Autoencoder").scale(0.8)
        subtitle = Text("(Analogy: The Master Summarizer)", font_size=24, color=GRAY).next_to(title, DOWN)
        self.play(Write(title), FadeIn(subtitle))
        self.wait(1)

        # --- Dynamic Architecture Construction ---
        
        # 1. Input (High Dimensional)
        input_vec = Rectangle(height=3.0, width=0.5, fill_color=COLOR_INPUT, fill_opacity=0.5)
        input_dots = VGroup(*[Dot(radius=0.05, color=WHITE).move_to(input_vec.get_center() + np.array([0, y, 0])) for y in np.linspace(-1.2, 1.2, 10)])
        input_lbl = Text("High-Res Data", font_size=16).next_to(input_vec, DOWN)
        input_grp = VGroup(input_vec, input_dots, input_lbl).to_edge(LEFT, buff=1.0)
        
        self.play(GrowFromCenter(input_vec), FadeIn(input_dots), Write(input_lbl))

        # 2. Encoder (The Funnel)
        enc_trap = Polygon(
            [-1.2, 1.5, 0], [0.5, 1.5, 0], [0.5, -1.5, 0], [-1.2, -1.5, 0],
            fill_color=COLOR_ENCODER, fill_opacity=0.5, color=WHITE
        ).scale(0.7)
        # Morphing shape to look like a funnel
        enc_funnel = Polygon(
            [-1.0, 1.5, 0], [1.0, 0.5, 0], [1.0, -0.5, 0], [-1.0, -1.5, 0],
            fill_color=COLOR_ENCODER, fill_opacity=0.5, color=WHITE
        ).scale(0.7).next_to(input_grp, RIGHT)
        
        enc_lbl = Text("Encoder", font_size=16).move_to(enc_funnel)
        
        self.play(Transform(enc_trap, enc_funnel), FadeIn(enc_lbl))

        # 3. Latent (The Compressed Essence)
        lat_vec = Rectangle(height=0.8, width=0.3, fill_color=COLOR_LATENT, fill_opacity=0.9)
        lat_lbl = Text("Latent Code", font_size=16, color=COLOR_LATENT).next_to(lat_vec, DOWN)
        lat_grp = VGroup(lat_vec, lat_lbl).next_to(enc_funnel, RIGHT)
        
        self.play(GrowFromCenter(lat_vec), Write(lat_lbl))

        # 4. Decoder (The Expander)
        dec_funnel = Polygon(
            [-1.0, 0.5, 0], [1.0, 1.5, 0], [1.0, -1.5, 0], [-1.0, -0.5, 0],
            fill_color=COLOR_DECODER, fill_opacity=0.5, color=WHITE
        ).scale(0.7).next_to(lat_grp, RIGHT)
        dec_lbl = Text("Decoder", font_size=16).move_to(dec_funnel)
        
        self.play(FadeIn(dec_funnel), FadeIn(dec_lbl))

        # 5. Output
        out_vec = Rectangle(height=3.0, width=0.5, fill_color=COLOR_OUTPUT, fill_opacity=0.5)
        out_lbl = Text("Reconstruction", font_size=16).next_to(out_vec, DOWN)
        out_grp = VGroup(out_vec, out_lbl).next_to(dec_funnel, RIGHT)
        
        self.play(GrowFromCenter(out_vec), Write(out_lbl))

        # Center the whole pipeline
        full_sys = VGroup(input_grp, enc_trap, enc_lbl, lat_grp, dec_funnel, dec_lbl, out_grp)
        self.play(full_sys.animate.scale_to_fit_width(config.frame_width - 1).move_to(UP*0.5))

        # --- Deep Dive Animation: The Compression Flow ---
        
        # Create a "Data Packet"
        packet = Circle(radius=0.2, color=WHITE, fill_opacity=1)
        packet.move_to(input_vec.get_center())
        
        # Animate flow
        self.play(packet.animate.move_to(enc_funnel.get_center()), run_time=1)
        
        # Squeeze animation at bottleneck
        self.play(
            packet.animate.move_to(lat_vec.get_center()).scale(0.3).set_color(COLOR_LATENT), 
            run_time=0.8
        )
        self.play(Indicate(lat_vec, scale_factor=1.2, color=YELLOW))
        
        # Expand animation
        self.play(
            packet.animate.move_to(dec_funnel.get_center()).scale(3.3).set_color(WHITE), 
            run_time=0.8
        )
        self.play(packet.animate.move_to(out_vec.get_center()), run_time=1)
        self.play(Flash(out_vec, color=GREEN, flash_radius=0.5))
        self.play(FadeOut(packet))

        # Explanation
        expl_text = VGroup(
            Text("By squeezing data through a bottleneck,", font_size=24, color=YELLOW),
            Text("we force the model to learn the 'Essence' rather than memorizing pixels.", font_size=20)
        ).arrange(DOWN).to_edge(DOWN)
        
        self.play(Write(expl_text))
        self.wait(3)
        self.play(FadeOut(full_sys), FadeOut(expl_text), FadeOut(title), FadeOut(subtitle))

    def part_2_latent_space_problem(self):
        title = Title("Part 2: The 'Dead Zone' Problem").scale(0.8)
        self.play(Write(title))

        # Graph setup
        plane = NumberPlane(x_range=[-3, 3], y_range=[-3, 3], background_line_style={"stroke_opacity": 0.3})
        plane.scale(0.7).to_edge(LEFT, buff=1.5)
        self.play(Create(plane))

        # Data points appearing sequentially
        dots_a = VGroup()
        for _ in range(8):
            dots_a.add(Dot(plane.coords_to_point(np.random.normal(-1.5, 0.2), np.random.normal(1, 0.2)), color=BLUE))
        
        dots_b = VGroup()
        for _ in range(8):
            dots_b.add(Dot(plane.coords_to_point(np.random.normal(1.5, 0.2), np.random.normal(-1, 0.2)), color=RED))

        lbl_a = Text("Digit '1'", font_size=16, color=BLUE).next_to(dots_a, UP)
        lbl_b = Text("Digit '0'", font_size=16, color=RED).next_to(dots_b, DOWN)

        self.play(LaggedStart(*[GrowFromCenter(d) for d in dots_a], lag_ratio=0.1))
        self.play(Write(lbl_a))
        self.play(LaggedStart(*[GrowFromCenter(d) for d in dots_b], lag_ratio=0.1))
        self.play(Write(lbl_b))

        # --- The Investigation ---
        
        # Create a "Probe"
        probe = Dot(color=WHITE, radius=0.1).move_to(plane.coords_to_point(-1.5, 1))
        probe_lbl = Text("Decoder Probe", font_size=14).next_to(probe, RIGHT)
        self.play(FadeIn(probe), FadeIn(probe_lbl))

        # Probe Known Area
        self.play(probe.animate.move_to(dots_a[0].get_center()))
        valid_box = RoundedRectangle(corner_radius=0.2, height=1, width=3).to_edge(RIGHT)
        valid_text = Text("Valid Output", color=GREEN).move_to(valid_box)
        self.play(Create(valid_box), Write(valid_text))
        self.wait(1)
        self.play(FadeOut(valid_box), FadeOut(valid_text))

        # Probe The Void
        dead_pt = plane.coords_to_point(0,0)
        self.play(probe.animate.move_to(dead_pt).set_color(YELLOW))
        
        # Garbage Output Animation
        garbage_box = RoundedRectangle(corner_radius=0.2, height=1, width=3, color=RED).to_edge(RIGHT)
        garbage_text = Text("GARBAGE NOISE", color=RED).move_to(garbage_box)
        
        # Shake effect
        self.play(Create(garbage_box), Write(garbage_text))
        self.play(Wiggle(garbage_box), Wiggle(probe))
        
        # Deep Explanation
        expl = Text(
            "Standard AEs are discontinuous.\nThe decoder has no idea what exists here.",
            font_size=20, color=GRAY
        ).next_to(garbage_box, DOWN)
        self.play(Write(expl))
        
        self.wait(3)
        self.play(FadeOut(Group(*self.mobjects)))

    def part_3_vae_concept(self):
        title = Title("Part 3: The VAE Solution").scale(0.8)
        subtitle = Text("(Analogy: Casting a Wide Net)", font_size=24, color=GRAY).next_to(title, DOWN)
        self.play(Write(title), Write(subtitle))

        # 1. The Transformation Animation
        
        # Start: Discrete Point
        start_pt = Dot(color=WHITE, radius=0.1).move_to(LEFT * 3)
        lbl_pt = Text("Point (Deterministic)", font_size=20).next_to(start_pt, DOWN)
        self.play(FadeIn(start_pt), Write(lbl_pt))
        self.wait(1)

        # Transform: Split into Mu/Sigma
        mu_dot = Dot(color=BLUE).move_to(LEFT * 3 + UP * 0.5)
        sigma_ring = Circle(radius=0.1, color=RED).move_to(LEFT * 3 + DOWN * 0.5)
        
        self.play(
            ReplacementTransform(start_pt, mu_dot),
            FadeIn(sigma_ring),
            FadeOut(lbl_pt)
        )
        
        lbl_mu = MathTex(r"\mu", color=BLUE).next_to(mu_dot, LEFT)
        lbl_sig = MathTex(r"\sigma", color=RED).next_to(sigma_ring, LEFT)
        self.play(Write(lbl_mu), Write(lbl_sig))

        # Transform: Expand to Cloud
        cloud = Ellipse(width=3.0, height=2.0, color=YELLOW, fill_opacity=0.3).move_to(RIGHT * 2)
        arrow = Arrow(mu_dot.get_right(), cloud.get_left(), buff=0.5)
        
        self.play(
            GrowArrow(arrow),
            ReplacementTransform(mu_dot.copy(), cloud),
            sigma_ring.animate.scale(5).move_to(cloud.get_center()).set_opacity(0)
        )
        
        lbl_cloud = Text("Probability Cloud", font_size=24, color=YELLOW).next_to(cloud, UP)
        self.play(Write(lbl_cloud))

        # --- Deep Dive: Sampling Animation ---
        # Show actual samples appearing
        
        sample_text = Text("Training Samples:", font_size=20).next_to(cloud, DOWN, buff=0.5)
        self.play(Write(sample_text))

        for i in range(5):
            # Random point in ellipse
            r = np.sqrt(np.random.random())
            theta = np.random.random() * 2 * np.pi
            x = 1.5 * r * np.cos(theta) + 2 # +2 to shift to cloud center
            y = 1.0 * r * np.sin(theta)
            
            s = Dot(point=[x,y,0], color=WHITE, radius=0.05)
            self.play(FadeIn(s, scale=0.5), run_time=0.2)
        
        # Explanation
        final_text = Text(
            "By sampling random points during training,\nwe force the decoder to understand the whole region.",
            font_size=20, color=GREEN
        ).to_edge(DOWN)
        self.play(Write(final_text))

        self.wait(3)
        self.play(FadeOut(Group(*self.mobjects)))

    def part_4_vae_mathematics_deep(self):
        title = Title("Part 4: The Math (Making it Tractable)").scale(0.8)
        self.play(Write(title))

        # --- Section 1: The Problem ---
        # Show the integral
        int_eq = MathTex(r"P(x) = \int P(x|z)P(z) dz", font_size=40)
        int_lbl = Text("The Marginal Likelihood", font_size=24, color=BLUE).next_to(int_eq, UP)
        
        self.play(Write(int_lbl), Write(int_eq))
        self.wait(1)
        
        # Show it's hard
        cross = Cross(int_eq).set_color(RED)
        fail_txt = Text("Intractable (Too many z's!)", font_size=24, color=RED).next_to(int_eq, DOWN)
        self.play(Create(cross), Write(fail_txt))
        self.wait(2)
        
        # Clear
        self.play(FadeOut(int_eq), FadeOut(int_lbl), FadeOut(cross), FadeOut(fail_txt))

        # --- Section 2: The Solution (ELBO) ---
        # We build the ELBO equation term by term
        
        elbo_title = Text("The Objective Function (ELBO)", font_size=28, color=GREEN).to_edge(UP, buff=1.5)
        self.play(Write(elbo_title))

        # Term 1: Reconstruction
        term_rec = MathTex(r"\mathbb{E}_{q}[\log p(x|z)]", font_size=36, color=BLUE).move_to(LEFT * 2)
        lbl_rec = Text("Reconstruction\n(Quality)", font_size=16, color=BLUE).next_to(term_rec, DOWN)
        
        self.play(Write(term_rec))
        self.play(FadeIn(lbl_rec))

        # Operator
        minus = MathTex("-", font_size=36).next_to(term_rec, RIGHT)
        self.play(Write(minus))

        # Term 2: KL Divergence
        term_kl = MathTex(r"D_{KL}(q(z|x) || p(z))", font_size=36, color=YELLOW).next_to(minus, RIGHT)
        lbl_kl = Text("Regularization\n(Structure)", font_size=16, color=YELLOW).next_to(term_kl, DOWN)
        
        self.play(Write(term_kl))
        self.play(FadeIn(lbl_kl))

        # --- Deep Dive: Visualizing the Tug-of-War ---
        
        # Setup balance scale visual
        bar = Line(LEFT*3, RIGHT*3, color=WHITE).shift(DOWN*1.5)
        pivot = Triangle(color=WHITE, fill_opacity=1).scale(0.2).next_to(bar, DOWN, buff=0)
        
        ball_rec = Circle(radius=0.5, color=BLUE, fill_opacity=0.8).next_to(bar, UP, buff=0).shift(LEFT*2)
        t_rec = Text("Accuracy", font_size=14).move_to(ball_rec)
        
        ball_kl = Circle(radius=0.5, color=YELLOW, fill_opacity=0.8).next_to(bar, UP, buff=0).shift(RIGHT*2)
        t_kl = Text("Smoothness", font_size=14).move_to(ball_kl)
        
        self.play(Create(bar), FadeIn(pivot), FadeIn(ball_rec), Write(t_rec), FadeIn(ball_kl), Write(t_kl))
        
        # Animation: Balancing
        self.play(
            Rotate(bar, angle=0.1, about_point=pivot.get_top()),
            ball_rec.animate.shift(DOWN*0.3),
            ball_kl.animate.shift(UP*0.3),
            run_time=1
        )
        self.play(
            Rotate(bar, angle=-0.2, about_point=pivot.get_top()),
            ball_rec.animate.shift(UP*0.6),
            ball_kl.animate.shift(DOWN*0.6),
            run_time=1
        )
        self.play(
            Rotate(bar, angle=0.1, about_point=pivot.get_top()),
            ball_rec.animate.shift(DOWN*0.3),
            ball_kl.animate.shift(UP*0.3),
            run_time=1
        )

        expl = Text("The network must balance accuracy vs. organized latent space.", font_size=20).to_edge(DOWN)
        self.play(Write(expl))
        self.wait(3)

        self.play(FadeOut(Group(*self.mobjects)))

    def part_5_reparameterization_deep(self):
        title = Title("Part 5: The Reparameterization Trick").scale(0.8)
        self.play(Write(title))

        # --- Phase 1: The Problem (Interactive Graph) ---
        
        p1_lbl = Text("Standard Sampling (Broken Gradient)", font_size=24, color=RED).to_edge(UP, buff=1.5)
        self.play(Write(p1_lbl))

        # Nodes
        mu = MathTex(r"\mu").shift(LEFT*2 + UP)
        sig = MathTex(r"\sigma").shift(LEFT*2 + DOWN)
        box = Square(color=PURPLE, side_length=1)
        box_lbl = Text("Random", font_size=14).move_to(box)
        z = MathTex("z").shift(RIGHT*2)
        
        arrows = VGroup(
            Arrow(mu.get_right(), box.get_left()),
            Arrow(sig.get_right(), box.get_left()),
            Arrow(box.get_right(), z.get_left())
        )
        
        graph_bad = VGroup(mu, sig, box, box_lbl, z, arrows).center()
        self.play(FadeIn(graph_bad))

        # Animate Gradient attempting to pass
        grad_dot = Dot(color=RED).move_to(z.get_center())
        path = Line(z.get_center(), box.get_right())
        
        self.play(MoveAlongPath(grad_dot, path), run_time=1)
        self.play(Indicate(box, color=RED), Flash(box, color=RED))
        
        # Stop sign
        stop = Text("STOP", color=RED, weight=BOLD).move_to(box)
        self.play(Transform(box_lbl, stop))
        
        fail_txt = Text("Cannot differentiate random noise!", font_size=20, color=RED).next_to(graph_bad, DOWN)
        self.play(Write(fail_txt))
        self.wait(2)
        
        self.play(FadeOut(graph_bad), FadeOut(grad_dot), FadeOut(fail_txt), FadeOut(p1_lbl), FadeOut(box_lbl))

        # --- Phase 2: The Solution (Interactive Graph) ---
        
        p2_lbl = Text("Reparameterization (Gradient Highway)", font_size=24, color=GREEN).to_edge(UP, buff=1.5)
        self.play(Write(p2_lbl))

        # Equation
        eq = MathTex(r"z = \mu + \sigma \odot \epsilon", font_size=36).next_to(p2_lbl, DOWN)
        self.play(Write(eq))

        # Visualizing the "Side Loading" of noise
        
        # Main deterministic path
        mu_new = MathTex(r"\mu", color=BLUE).move_to(LEFT*2)
        sig_new = MathTex(r"\sigma", color=RED).move_to(LEFT*2 + DOWN*1.5)
        
        op_add = Circle(radius=0.3, color=WHITE).move_to(ORIGIN)
        op_plus = MathTex("+").move_to(op_add)
        
        op_mul = Circle(radius=0.3, color=WHITE).move_to(DOWN*1.5)
        op_times = MathTex(r"\times").move_to(op_mul)
        
        z_new = MathTex("z").move_to(RIGHT*2)
        
        # Noise from side
        eps = MathTex(r"\epsilon", color=COLOR_NOISE).move_to(LEFT*0.5 + UP*1.5)
        eps_lbl = Text("Fixed Noise", font_size=12, color=COLOR_NOISE).next_to(eps, UP)
        
        # Wires
        w1 = Arrow(mu_new.get_right(), op_add.get_left())
        w2 = Arrow(sig_new.get_right(), op_mul.get_left())
        w3 = Arrow(eps.get_bottom(), op_mul.get_top()) # Side load
        w4 = Arrow(op_mul.get_top(), op_add.get_bottom())
        w5 = Arrow(op_add.get_right(), z_new.get_left())
        
        graph_good = VGroup(mu_new, sig_new, eps, eps_lbl, op_add, op_plus, op_mul, op_times, z_new, w1, w2, w3, w4, w5).center().shift(DOWN*0.5)
        
        self.play(FadeIn(graph_good))
        
        # --- Deep Dive: Visualizing Success ---
        
        # Animate Gradient Flow (The Green Path)
        grad_dot_green = Dot(color=GREEN).move_to(z_new.get_center())
        
        # Trace path back to Mu
        trace_mu = TracedPath(grad_dot_green.get_center, stroke_color=GREEN, stroke_width=4, dissipating_time=1)
        self.add(trace_mu)
        
        self.play(grad_dot_green.animate.move_to(mu_new.get_center()), run_time=1.5)
        self.play(Indicate(mu_new, color=GREEN))
        
        # Trace path back to Sigma
        grad_dot_green.move_to(z_new.get_center())
        self.play(grad_dot_green.animate.move_to(op_mul.get_center()), run_time=0.5)
        self.play(grad_dot_green.animate.move_to(sig_new.get_center()), run_time=1.0)
        self.play(Indicate(sig_new, color=GREEN))
        
        success_txt = Text("Parameters are now updated successfully!", font_size=20, color=GREEN).to_edge(DOWN)
        self.play(Write(success_txt))

        self.wait(3)
        self.play(FadeOut(Group(*self.mobjects)))


%manim -qk -v warning AutoencoderDeepDive

Manim Community v0.19.0